# CuPL Oxford-IIIT Pet: đánh giá bằng prompt do tác giả công bố

Notebook này dùng **prompt Oxford Pets trong repository CuPL của tác giả**, không dùng `Semester/attributes/*` của Gemma. Nó chạy CLIP ViT-B/32 trên feature test cache của dự án và tính accuracy raw/TTA. Đây là **phép đánh giá lại tại chỗ** với backbone và cache của nhóm; không tự nhận là con số gốc trong paper CuPL hay kết quả Stanford Dogs.

Nguồn prompt: https://github.com/sarahpratt/CuPL/tree/main/all_prompts/full_prompts (tệp `pets_prompts_full.json`). Theo CuPL, các mô tả được sinh bởi LLM, CLIP mã hóa từng câu, trung bình vector cho mỗi lớp rồi chuẩn hóa. Máy chạy notebook cần truy cập GitHub một lần để tải prompt, hoặc tải thủ công tệp JSON theo URL và đặt vào `notebooks/paper_test/external/`. Không có số kết quả nào được điền sẵn.

In [ ]:
from pathlib import Path
import json, hashlib, urllib.request, urllib.error, re
import torch
import torch.nn.functional as F
from transformers import CLIPModel, CLIPTokenizer

def find_repo(start=Path.cwd()):
    for p in (start, *start.parents, Path("/content/CLIP-Animal-Classifier")):
        if (p / "paper/paper_CLIP_Gemma4_revised_v2.md").exists():
            return p
    raise FileNotFoundError("Mở notebook trong repository")
ROOT = find_repo()
EXT = ROOT / "notebooks/paper_test/external"
OUT = ROOT / "notebooks/paper_test/results"
EXT.mkdir(parents=True, exist_ok=True); OUT.mkdir(parents=True, exist_ok=True)
SOURCE_URL = "https://raw.githubusercontent.com/sarahpratt/CuPL/main/all_prompts/full_prompts/pets_prompts_full.json"
PROMPTS_PATH = EXT / "pets_prompts_full.json"
if not PROMPTS_PATH.exists():
    try:
        urllib.request.urlretrieve(SOURCE_URL, PROMPTS_PATH)
    except (urllib.error.URLError, OSError) as exc:
        raise RuntimeError(f"Không tải được prompt. Tải thủ công từ {SOURCE_URL} và đặt vào {PROMPTS_PATH}") from exc
prompt_bytes = PROMPTS_PATH.read_bytes()
prompt_sha256 = hashlib.sha256(prompt_bytes).hexdigest()
bank = json.loads(prompt_bytes)
print("CuPL released prompt SHA256:", prompt_sha256)
print("JSON type:", type(bank).__name__, "classes:", len(bank))
print("First keys:", list(bank)[:5] if isinstance(bank,dict) else "not a dictionary")

In [2]:
OX = ROOT / "Semester/cached_features/oxford_pets"
classes = json.loads((OX / "classes.json").read_text(encoding="utf-8"))
def canonical(s):
    return re.sub(r"[^a-z0-9]", "", str(s).lower())
def flatten_texts(value):
    if isinstance(value,str): return [value]
    if isinstance(value,list): return [x for item in value for x in flatten_texts(item)]
    if isinstance(value,dict): return [x for item in value.values() for x in flatten_texts(item)]
    raise TypeError(f"Không nhận diện cấu trúc prompt: {type(value)}")
if not isinstance(bank,dict):
    raise TypeError("Cần JSON dạng {class_name: [prompt,...]}; kiểm cấu trúc file tác giả")
by_class = {canonical(k):flatten_texts(v) for k,v in bank.items()}
missing = [c for c in classes if canonical(c) not in by_class]
if missing:
    raise ValueError(f"Không ghép được tên lớp {missing[:8]}; xem keys của JSON và thiết lập mapping tường minh, không tự đoán")
prompts = [by_class[canonical(c)] for c in classes]
assert all(p and all(isinstance(s,str) and s.strip() for s in p) for p in prompts)
print("Classes matched",len(prompts),"prompts/class min/max",min(map(len,prompts)),max(map(len,prompts)))
print("Example class:",classes[0],"example prompt:",prompts[0][0])

Classes matched 37 prompts/class min/max 20 20
Example class: Abyssinian example prompt: An Abyssinian looks like a small, slender cat with a short, fine coat that comes in many colors, including red, ruddy, blue, and fawn.


In [3]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
MODEL_ID = "openai/clip-vit-base-patch32"
tokenizer = CLIPTokenizer.from_pretrained(MODEL_ID)
clip = CLIPModel.from_pretrained(MODEL_ID).to(device).eval()
for par in clip.parameters(): par.requires_grad_(False)
weights = []
with torch.no_grad():
    for class_name, texts in zip(classes,prompts):
        features = []
        for start in range(0,len(texts),16):
            inputs = tokenizer(texts[start:start+16],padding=True,truncation=True,max_length=77,return_tensors="pt").to(device)
            features.append(F.normalize(clip.get_text_features(**inputs),dim=-1).cpu())
        class_weight = F.normalize(torch.cat(features).mean(dim=0,keepdim=True),dim=-1)
        weights.append(class_weight)
W = torch.cat(weights).to(device)
labels = torch.load(OX / "test_labels.pt",map_location="cpu",weights_only=True).long()
raw = torch.load(OX / "test_clip.pt",map_location="cpu",weights_only=True).float()
tta = torch.load(ROOT / "dataset/cached/test_features_tta.pt",map_location="cpu",weights_only=True).float()
assert raw.shape==(3669,512) and tta.shape==(3669,512) and len(labels)==3669 and W.shape==(37,512)
preds = {}
for name,features in [("raw",raw),("tta",tta)]:
    with torch.no_grad():
        logits = F.normalize(features.to(device),dim=-1) @ W.T
        preds[name] = logits.argmax(dim=-1).cpu()
    correct = int((preds[name]==labels).sum())
    print(name,correct,"/",len(labels),"=",f"{100*correct/len(labels):.4f}%")
print("Thiết bị:",device,"; TTA là cache của nhóm, cần mô tả năm view khi báo cáo.")

raw 3200 / 3669 = 87.2172%
tta 3215 / 3669 = 87.6261%
Thiết bị: mps ; TTA là cache của nhóm, cần mô tả năm view khi báo cáo.


In [4]:
metrics = {name:{"correct":int((pred==labels).sum()),"total":len(labels),
                 "accuracy":100*int((pred==labels).sum())/len(labels)} for name,pred in preds.items()}
report = {"method":"CuPL full prompts released by authors; local CLIP feature evaluation",
          "dataset":"Oxford-IIIT Pet official 3669 test","prompt_source_url":SOURCE_URL,
          "prompt_sha256":prompt_sha256,"model_id":MODEL_ID,
          "feature_files":{"raw":"Semester/cached_features/oxford_pets/test_clip.pt",
                           "tta":"dataset/cached/test_features_tta.pt"},
          "prompt_count_per_class":{c:len(p) for c,p in zip(classes,prompts)},
          "results":metrics}
(OUT / "cupl_official_prompts_oxford.json").write_text(json.dumps(report,indent=2,ensure_ascii=False),encoding="utf-8")
torch.save({"test_labels":labels,"pred_raw":preds["raw"],"pred_tta":preds["tta"],"classes":classes},
           OUT / "cupl_official_prompts_oxford_predictions.pt")
print("Đã lưu kết quả trong notebooks/paper_test/results/. Không thay số CuPL trong paper trước khi kiểm prompt bank, backbone, cache và protocol.")

Đã lưu kết quả trong notebooks/paper_test/results/. Không thay số CuPL trong paper trước khi kiểm prompt bank, backbone, cache và protocol.
